# 5.1 Drehmatrizen

Bisher haben wir Matrizen eingesetzt, um lineare Gleichungssysteme zu lösen
und Kräfte in Fachwerken zu berechnen. In diesem Kapitel entdecken wir eine
weitere Anwendung: Mit einer einzigen Matrixmultiplikation können wir ein
Objekt im Raum drehen. Unser Beispiel ist ein Dreieck, das wir zunächst in
der Ebene und dann im dreidimensionalen Raum rotieren.

## Lernziele

* [ ] Sie können die 2D-Drehmatrix $\mathbf{R}(\varphi)$ aus einem Drehwinkel
  aufbauen und auf einen Punkt anwenden.
* [ ] Sie wissen, was eine **orthogonale Matrix** ist, und können die
  Eigenschaft $\mathbf{R}^\top \mathbf{R} = \mathbf{I}$ numerisch überprüfen.
* [ ] Sie können die drei Grunddrehmatrizen $\mathbf{R}_x$, $\mathbf{R}_y$,
  $\mathbf{R}_z$ für Drehungen im 3D-Raum aufbauen.
* [ ] Sie wissen, was **Kardan-Winkel** sind, und können eine Gesamtrotation
  als Verkettung von Grunddrehungen berechnen.

## Drehmatrizen in 2D

Wir definieren ein Dreieck durch drei Eckpunkte in der $xy$-Ebene und stellen es
grafisch dar. Jede Zeile des Arrays enthält die $x$- und $y$-Koordinate eines
Eckpunkts.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import Polygon as MplPolygon

# --- Dreieck: drei Eckpunkte, je eine Zeile ---
# Spalte 0: x-Koordinate, Spalte 1: y-Koordinate
dreieck = np.array([
    [0.0, 0.0],   # Eckpunkt A
    [2.0, 0.0],   # Eckpunkt B
    [1.0, 2.0],   # Eckpunkt C
])

# --- Darstellung ---
fig, ax = plt.subplots(figsize=(4, 4))
ax.add_patch(MplPolygon(dreieck, closed=True, fill=False,
                         edgecolor='blue', linewidth=2))

# Eckpunkte beschriften
for label, punkt in zip(['A', 'B', 'C'], dreieck):
    ax.annotate(label, punkt, textcoords='offset points',
                xytext=(6, 4), fontsize=11)

ax.set_xlim(-3, 3); ax.set_ylim(-3, 3)
ax.set_aspect('equal')
ax.axhline(0, color='gray', linewidth=0.8)
ax.axvline(0, color='gray', linewidth=0.8)
ax.grid(True)
ax.set_title('Unser Dreieck')
plt.show()

Nun drehen wir Eckpunkt B um den Winkel $\varphi = 90°$ gegen den Uhrzeigersinn
um den Ursprung. Die **2D-Drehmatrix** für diesen Winkel lautet:

\begin{equation*}
\mathbf{R}(\varphi) = \begin{pmatrix} \cos\varphi & -\sin\varphi \\
\sin\varphi & \cos\varphi \end{pmatrix}.
\end{equation*}

Wir rechnen die Drehung von Eckpunkt B $= (2, 0)^\top$ um $\varphi = 90°$ von
Hand durch. Mit $\cos 90° = 0$ und $\sin 90° = 1$ ergibt sich:

\begin{equation*}
\mathbf{R}(90°) \cdot \vec{p}_B
= \begin{pmatrix} \cos 90° & -\sin 90° \\ \sin 90° & \cos 90° \end{pmatrix}
  \begin{pmatrix} 2 \\ 0 \end{pmatrix}
= \begin{pmatrix} 0 & -1 \\ 1 & 0 \end{pmatrix}
  \begin{pmatrix} 2 \\ 0 \end{pmatrix}
= \begin{pmatrix} 0 \cdot 2 + (-1) \cdot 0 \\ 1 \cdot 2 + 0 \cdot 0 \end{pmatrix}
= \begin{pmatrix} 0 \\ 2 \end{pmatrix}.
\end{equation*}

Punkt B wandert von $(2, 0)$ nach $(0, 2)$: er bewegt sich auf einem Kreisbogen
von der positiven $x$-Achse zur positiven $y$-Achse. Für Eckpunkt C $= (1, 2)^\top$
ergibt dieselbe Rechnung:

\begin{equation*}
\mathbf{R}(90°) \cdot \vec{p}_C
= \begin{pmatrix} 0 & -1 \\ 1 & 0 \end{pmatrix}
  \begin{pmatrix} 1 \\ 2 \end{pmatrix}
= \begin{pmatrix} 0 \cdot 1 + (-1) \cdot 2 \\ 1 \cdot 1 + 0 \cdot 2 \end{pmatrix}
= \begin{pmatrix} -2 \\ 1 \end{pmatrix}.
\end{equation*}

In [ ]:
# --- 2D-Drehmatrix ---
# phi: Drehwinkel in Radiant (positiv = gegen den Uhrzeigersinn)
def drehmatrix_2d(phi):
    return np.array([
        [np.cos(phi), -np.sin(phi)],
        [np.sin(phi),  np.cos(phi)],
    ])

# --- Drehwinkel: 90 Grad ---
# np.deg2rad rechnet Grad in Radiant um (NumPy erwartet Radiant)
phi = np.deg2rad(90)
R = drehmatrix_2d(phi)

print("Drehmatrix R(90°):")
print(np.round(R, 4))

# --- Eckpunkt B drehen: R @ punkt ---
# @ ist der Matrixmultiplikations-Operator
punkt_B = dreieck[1]
punkt_B_gedreht = R @ punkt_B

print(f"\nEckpunkt B original: {punkt_B}")
print(f"Eckpunkt B gedreht:  {np.round(punkt_B_gedreht, 4)}")

Hinweis: NumPy-Funktionen wie `np.cos` und `np.sin` erwarten Winkel in Radiant,
nicht in Grad. Die Umrechnung erfolgt mit `np.deg2rad`. Ein voller Kreis
entspricht $2\pi \approx 6.28$ Radiant.

### Mini-Übung 1

1. Berechnen Sie die gedrehten Koordinaten aller drei Eckpunkte für
   $\varphi = 90°$ in Python.
2. Zeichnen Sie das gedrehte Dreieck in dieselbe Abbildung wie das
   ursprüngliche. Tipp: Berechnen Sie alle drei gedrehten Eckpunkte
   und speichern Sie sie in einem Array `dreieck_gedreht`. Mit
   `(R @ dreieck.T).T` drehen Sie alle Punkte gleichzeitig.
3. Was ergibt sich, wenn Sie das gedrehte Dreieck nochmals um 90° drehen?
   Begründen Sie in einem Satz, ohne den Code auszuführen.

In [ ]:
# Code-Zelle

## Die Drehmatrix ist orthogonal

Eine **orthogonale Matrix** erfüllt $\mathbf{R}^\top \mathbf{R} =
\mathbf{I}$, wobei $\mathbf{I}$ die Einheitsmatrix ist. Das bedeutet: die
Transponierte ist gleichzeitig die Inverse, also $\mathbf{R}^{-1} =
\mathbf{R}^\top$. Außerdem gilt für jede Drehmatrix $\det(\mathbf{R}) = 1$.

In [ ]:
phi = np.deg2rad(90)
R = drehmatrix_2d(phi)

# --- Orthogonalitätsprüfung: R^T * R soll die Einheitsmatrix ergeben ---
RtR = R.T @ R
print("R^T * R =")
print(np.round(RtR, 10))

# --- Determinante ---
det_R = np.linalg.det(R)
print(f"\ndet(R) = {det_R:.6f}")

# --- np.allclose prüft näherungsweise Gleichheit ---
# Werte wie cos(90°) lassen sich nicht exakt als Dezimalzahl darstellen,
# daher kleine Rundungsfehler (z.B. 6.12e-17 statt 0.0)
print(f"R^T * R ≈ I: {np.allclose(RtR, np.eye(2))}")

Die Eigenschaft $\mathbf{R}^\top \mathbf{R} = \mathbf{I}$ garantiert, dass
eine Drehung Längen und Winkel erhält. Ein gedrehtes Dreieck ist also
kongruent zum Original. Die praktische Konsequenz: um eine Drehung
rückgängig zu machen, genügt es, mit der Transponierten zu multiplizieren,
also $\mathbf{R}^\top \cdot \vec{p}_\text{gedreht} = \vec{p}_\text{original}$.

Wir überprüfen dies von Hand für $\varphi = 90°$. Die Transponierte lautet:

\begin{equation*}
\mathbf{R}(90°)^\top
= \begin{pmatrix} 0 & 1 \\ -1 & 0 \end{pmatrix}.
\end{equation*}

Wir wenden sie auf den gedrehten Punkt B $= (0, 2)^\top$ an und erhalten
erwartungsgemäß den Ausgangspunkt zurück:

\begin{equation*}
\mathbf{R}(90°)^\top \cdot \vec{p}_{B,\text{gedreht}}
= \begin{pmatrix} 0 & 1 \\ -1 & 0 \end{pmatrix}
  \begin{pmatrix} 0 \\ 2 \end{pmatrix}
= \begin{pmatrix} 0 \cdot 0 + 1 \cdot 2 \\ (-1) \cdot 0 + 0 \cdot 2 \end{pmatrix}
= \begin{pmatrix} 2 \\ 0 \end{pmatrix}
= \vec{p}_B.
\end{equation*}

### Mini-Übung 2

1. Berechnen Sie $\mathbf{R}(90°)^\top \mathbf{R}(90°)$ von Hand, indem
   Sie die Matrizenmultiplikation vollständig ausschreiben. Was ergibt sich?
2. Drehen Sie Eckpunkt C nach der 90°-Drehung mit der Transponierten zurück.
   Verifizieren Sie das Ergebnis mit Python.
3. Warum reicht es für die Rückdrehung, die Matrix zu transponieren, statt
   die Inverse zu berechnen? Formulieren Sie die Antwort in einem Satz ohne
   Code.

In [ ]:
# Code-Zelle

## Drehmatrizen in 3D: Rollen, Nicken, Gieren

Im dreidimensionalen Raum gibt es drei unabhängige Drehachsen. Wir
verwenden die **Kardan-Winkel** (auch Tait-Bryan-Winkel), die in der
Technischen Mechanik und der Fahrzeugtechnik Standard sind. Jede
Grunddrehung dreht um genau eine Koordinatenachse.

* **Gieren** (Yaw, Winkel $\alpha$): Drehung um die $z$-Achse, wie ein
Fahrzeug, das nach links oder rechts lenkt.
* **Nicken** (Pitch, Winkel $\beta$): Drehung um die $y$-Achse, wie ein
Fahrzeug, das eine Steigung hinauffährt.
* **Rollen** (Roll, Winkel $\gamma$): Drehung um die $x$-Achse, wie ein
Fahrzeug, das sich zur Seite neigt.

In [ ]:
# --- Grunddrehmatrizen in 3D ---

def R_gieren(alpha):
    """Drehung um die z-Achse (Gieren / Yaw), Winkel alpha"""
    return np.array([
        [ np.cos(alpha), -np.sin(alpha), 0],
        [ np.sin(alpha),  np.cos(alpha), 0],
        [             0,              0, 1],
    ])

def R_nicken(beta):
    """Drehung um die y-Achse (Nicken / Pitch), Winkel beta"""
    return np.array([
        [ np.cos(beta), 0, np.sin(beta)],
        [            0, 1,            0],
        [-np.sin(beta), 0, np.cos(beta)],
    ])

def R_rollen(gamma):
    """Drehung um die x-Achse (Rollen / Roll), Winkel gamma"""
    return np.array([
        [1,              0,               0],
        [0,  np.cos(gamma), -np.sin(gamma)],
        [0,  np.sin(gamma),  np.cos(gamma)],
    ])

# --- Kardan-Winkel: Gesamtrotation R = R_rollen * R_nicken * R_gieren ---
# Reihenfolge (von rechts gelesen): zuerst Gieren, dann Nicken, dann Rollen
alpha = np.deg2rad(30)   # Gieren:  30 Grad
beta  = np.deg2rad(20)   # Nicken:  20 Grad
gamma = np.deg2rad(10)   # Rollen:  10 Grad

R_gesamt = R_rollen(gamma) @ R_nicken(beta) @ R_gieren(alpha)

print("Gesamtrotation R (Kardan Z-Y-X):")
print(np.round(R_gesamt, 4))

# --- Einen 3D-Punkt drehen ---
# Eckpunkt B aus unserem Dreieck, in die Ebene z=0 eingebettet
punkt_B_3d = np.array([2.0, 0.0, 0.0])
punkt_B_gedreht = R_gesamt @ punkt_B_3d

print(f"\nPunkt B (3D, z=0): {punkt_B_3d}")
print(f"Punkt B gedreht:   {np.round(punkt_B_gedreht, 4)}")

Die Gesamtrotationsmatrix $\mathbf{R} = \mathbf{R}_x(\gamma) \cdot
\mathbf{R}_y(\beta) \cdot \mathbf{R}_z(\alpha)$ beschreibt die kombinierte
Wirkung der drei Grunddrehungen. Die Reihenfolge ist nicht egal: zuerst
wird die rechtsstehende Matrix angewendet, also zuerst Gieren, dann Nicken,
dann Rollen.

Um ein Gefühl für die einzelnen Schritte zu entwickeln, rechnen wir die
Gesamtdrehung von Punkt B $= (2, 0, 0)^\top$ schrittweise durch. Mit
$\cos 30° = \frac{\sqrt{3}}{2} \approx 0.866$ und $\sin 30° = 0.5$ ergibt
sich für den ersten Schritt:

**Schritt 1:** Gieren um $\alpha = 30°$ (Drehung in der $xy$-Ebene)

\begin{equation*}
\vec{p}^{(1)} = \mathbf{R}_z(30°)\,\vec{p}_B
= \begin{pmatrix}
    \cos 30° & -\sin 30° & 0 \\
    \sin 30° &  \cos 30° & 0 \\
    0        &  0        & 1
  \end{pmatrix}
  \begin{pmatrix} 2 \\ 0 \\ 0 \end{pmatrix}
= \begin{pmatrix} 0.866 \cdot 2 \\ 0.5 \cdot 2 \\ 0 \end{pmatrix}
= \begin{pmatrix} 1.732 \\ 1.000 \\ 0 \end{pmatrix}.
\end{equation*}

Nach dem Gieren liegt Punkt B noch in der $xy$-Ebene ($z = 0$). Die beiden
folgenden Schritte liefern analog:

**Schritt 2:** Nicken um $\beta = 20°$ $\;\Rightarrow\;$
$\vec{p}^{(2)} = \mathbf{R}_y(20°)\,\vec{p}^{(1)} \approx (1.628,\; 1.000,\; {-0.593})^\top$

**Schritt 3:** Rollen um $\gamma = 10°$ $\;\Rightarrow\;$
$\vec{p}^{(3)} = \mathbf{R}_x(10°)\,\vec{p}^{(2)} \approx (1.628,\; 1.088,\; {-0.410})^\top$

Das Ergebnis stimmt mit der Python-Ausgabe oben überein. Erst das Nicken
(Schritt 2) hebt Punkt B aus der $xy$-Ebene heraus; das Rollen (Schritt 3)
verschiebt anschließend nur noch die $y$- und $z$-Komponente.

### Mini-Übung 3

1. Setzen Sie $\alpha = 90°$, $\beta = 0°$, $\gamma = 0°$. Welche Matrix
   ergibt `R_gesamt`? Vergleichen Sie mit `R_gieren(np.deg2rad(90))`.
2. Drehen Sie Punkt B mit reinem Gieren ($\alpha = 45°$, $\beta = \gamma
   = 0°$). Liegt das Ergebnis noch in der $xy$-Ebene (d.h. ist $z = 0$)?
   Begründen Sie in einem Satz, ohne den Code auszuführen.
3. Tauschen Sie die Reihenfolge: `R_gieren(alpha) @ R_nicken(beta) @
   R_rollen(gamma)`. Ergibt sich für $\alpha = 30°$, $\beta = 20°$,
   $\gamma = 10°$ dasselbe Ergebnis? Was folgt daraus?

In [ ]:
# Code-Zelle

## Zusammenfassung und Ausblick

Wir haben die 2D-Drehmatrix $\mathbf{R}(\varphi)$ kennengelernt und gesehen,
dass sie einen Punkt durch eine einfache Matrix-Vektor-Multiplikation dreht.
Die Eigenschaft $\mathbf{R}^\top \mathbf{R} = \mathbf{I}$ garantiert, dass
Längen und Winkel erhalten bleiben, und macht die Berechnung der Rückdrehung
besonders einfach. Im 3D-Raum beschreiben die Kardan-Winkel (Gieren, Nicken,
Rollen) jede beliebige Orientierung als Verkettung von drei Grunddrehungen.

Noch fehlt uns ein Werkzeug: Die Drehmatrix kann drehen, aber nicht
verschieben. Im nächsten Abschnitt lernen wir die **homogenen Koordinaten**
kennen, mit denen wir Drehung und Translation in einer einzigen
Matrixmultiplikation zusammenfassen.